# Partición en dos subconjuntos

Repartir una lista de números positivos en dos grupos con la misma suma (o con la menor diferencia posible). Si la suma total es $S$, basta con encontrar un subconjunto que sume $\lfloor S/2 \rfloor$ o lo más cerca posible por debajo: es la suma de subconjuntos con objetivo $S/2$. El conjunto de sumas alcanzables con los $i$ primeros números cumple

$$A_0 = \{0\}, \qquad A_i = A_{i-1} \cup \{s + a_i : s \in A_{i-1}\}$$

y cuesta $O(n \cdot S)$ en lugar de los $O(2^n)$ repartos posibles.

## 1. ¿Se puede partir en dos mitades iguales?

Decidirlo con el conjunto de sumas alcanzables y también con un entero usado como conjunto de bits (el bit $s$ está a 1 si la suma $s$ es alcanzable, y añadir $a$ es `bits | bits << a`). Comprobar ambas versiones contra fuerza bruta.

In [1]:
import random
from itertools import combinations

def particion_conjunto(numeros):
    total = sum(numeros)
    if total % 2:
        return False
    alcanzables = {0}
    for a in numeros:
        alcanzables |= {s + a for s in alcanzables if s + a <= total // 2}
    return total // 2 in alcanzables

def particion_bits(numeros):
    total, bits = sum(numeros), 1
    for a in numeros:
        bits |= bits << a
    return total % 2 == 0 and bits >> (total // 2) & 1 == 1

def particion_fuerza_bruta(numeros):
    return any(2 * sum(c) == sum(numeros) for r in range(len(numeros) + 1) for c in combinations(numeros, r))

random.seed(1)
for _ in range(1000):
    numeros = [random.randint(1, 20) for _ in range(random.randint(0, 10))]
    assert particion_conjunto(numeros) == particion_bits(numeros) == particion_fuerza_bruta(numeros)
print(particion_bits([1, 5, 11, 5]), particion_bits([1, 2, 3, 5]))

True False


## 2. Reconstruir los dos grupos

Guardar, para cada suma alcanzable, el último número que se usó para llegar a ella por primera vez y de qué suma venía. Devolver los dos grupos de la partición de menor diferencia y comprobar que juntos son la lista original.

In [2]:
from collections import Counter

def mejor_particion(numeros):
    total = sum(numeros)
    previo = {0: None}  # suma -> (índice del número, suma anterior)
    for i, a in enumerate(numeros):
        for s in list(previo):
            if s + a <= total // 2 and s + a not in previo:
                previo[s + a] = (i, s)
    s = max(previo)
    indices = set()
    while previo[s] is not None:
        i, s = previo[s]
        indices.add(i)
    grupo1 = [a for i, a in enumerate(numeros) if i in indices]
    grupo2 = [a for i, a in enumerate(numeros) if i not in indices]
    return grupo1, grupo2

grupo1, grupo2 = mejor_particion([3, 1, 4, 2, 2, 1])
print(grupo1, grupo2, sum(grupo1), sum(grupo2))

random.seed(2)
for _ in range(500):
    numeros = [random.randint(1, 20) for _ in range(random.randint(0, 10))]
    g1, g2 = mejor_particion(numeros)
    assert Counter(g1) + Counter(g2) == Counter(numeros)
    assert sum(g1) <= sum(g2) and particion_fuerza_bruta(numeros) == (sum(g1) == sum(g2))

[3, 1, 2] [4, 2, 1] 6 7


## 3. Diferencia mínima

Calcular la menor diferencia posible $S - 2s$ entre las sumas de los dos grupos, donde $s$ es la mayor suma alcanzable que no pasa de $S/2$, y comprobarla contra fuerza bruta y contra la reconstrucción anterior.

In [3]:
def diferencia_minima(numeros):
    total, bits = sum(numeros), 1
    for a in numeros:
        bits |= bits << a
    mitad = bits & ((1 << (total // 2 + 1)) - 1)  # se queda con las sumas <= S/2
    return total - 2 * (mitad.bit_length() - 1)

def diferencia_fuerza_bruta(numeros):
    return min(abs(sum(numeros) - 2 * sum(c)) for r in range(len(numeros) + 1) for c in combinations(numeros, r))

random.seed(3)
for _ in range(1000):
    numeros = [random.randint(1, 50) for _ in range(random.randint(0, 10))]
    g1, g2 = mejor_particion(numeros)
    assert diferencia_minima(numeros) == diferencia_fuerza_bruta(numeros) == sum(g2) - sum(g1)
print(diferencia_minima([1, 6, 11, 5]), diferencia_minima([10, 20, 15, 5, 25]), diferencia_minima([7]))

1 5 7


## 4. Contar las particiones iguales

Contar de cuántas formas se puede partir la lista en dos grupos de igual suma, sin distinguir qué grupo es el primero: es el número de subconjuntos que suman $S/2$ dividido entre 2. Comprobarlo contra fuerza bruta y contar las particiones de los números del 1 al 20.

In [4]:
def cuenta_particiones(numeros):
    total = sum(numeros)
    if total % 2:
        return 0
    N = [1] + [0] * (total // 2)
    for a in numeros:
        for s in range(total // 2, a - 1, -1):
            N[s] += N[s - a]
    return N[total // 2] // 2

def cuenta_fuerza_bruta(numeros):
    return sum(2 * sum(c) == sum(numeros) for r in range(len(numeros) + 1) for c in combinations(numeros, r)) // 2

random.seed(4)
for _ in range(500):
    numeros = [random.randint(1, 10) for _ in range(random.randint(1, 12))]
    assert cuenta_particiones(numeros) == cuenta_fuerza_bruta(numeros)
print(cuenta_particiones([1, 5, 11, 5]), cuenta_particiones(list(range(1, 21))))

1 7636


## 5. Repartir tareas entre dos procesadores

Hay que repartir tareas de distinta duración entre dos procesadores para que acaben lo antes posible: el tiempo total es la suma del grupo mayor. Comparar el reparto óptimo con la heurística voraz que ordena las tareas de mayor a menor y da cada una al procesador menos cargado. Medir en 2000 casos aleatorios cuántas veces falla la voraz y cuánto se aleja como máximo.

In [5]:
def voraz(tareas):
    cargas = [0, 0]
    for t in sorted(tareas, reverse=True):
        cargas[cargas.index(min(cargas))] += t
    return max(cargas)

def optimo(tareas):
    return (sum(tareas) + diferencia_minima(tareas)) // 2

print(voraz([3, 3, 2, 2, 2]), optimo([3, 3, 2, 2, 2]))

random.seed(5)
fallos, peor = 0, 1
for _ in range(2000):
    tareas = [random.randint(1, 30) for _ in range(random.randint(2, 12))]
    v, o = voraz(tareas), optimo(tareas)
    assert v >= o
    fallos += v > o
    peor = max(peor, v / o)
print(f"la voraz no es óptima en {fallos} de 2000 casos; como mucho un {peor - 1:.1%} peor")

7 6
la voraz no es óptima en 671 de 2000 casos; como mucho un 12.0% peor
